In [ ]:
!pip install requests beautifulsoup4 pandas lxml -q

In [ ]:
import re
import json
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [ ]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
]

ARTICLE_URL_PATTERN = re.compile(
    r"^https?://vnexpress\.net/.+?\.html(?:\?.*)?$",
    re.IGNORECASE
)

NUMBER_PATTERN = re.compile(
    r"""
    \b\d{1,2}/\d{1,2}/\d{2,4}\b
    |
    \b\d{1,3}(?:[.,]\d{3})*(?:[.,]\d+)?\s?(?:%|tỷ|triệu|nghìn|ngàn|USD|usd|đồng|cổ\sphiếu|ha|năm|tháng)?\b
    """,
    re.VERBOSE | re.IGNORECASE,
)

def random_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

def clean_whitespace(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

def normalize_text(text):
    text = clean_whitespace(text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip()

def word_count(text):
    return len(text.split())

def extract_numbers(text):
    return [normalize_text(x) for x in NUMBER_PATTERN.findall(text)]

def is_valid_article_url(url):
    if not ARTICLE_URL_PATTERN.match(url):
        return False

    bad_keywords = [
        "/video/",
        "/podcast/",
        "/infographics/",
        "/interactive/",
        "/photo/",
    ]
    if any(x in url.lower() for x in bad_keywords):
        return False

    return True

In [ ]:
def get_article_links(list_url, max_links=100):
    res = requests.get(list_url, headers=random_headers(), timeout=20)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "lxml")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        full_url = urljoin(list_url, href)

        if is_valid_article_url(full_url):
            links.add(full_url.split("?")[0])

    return sorted(links)[:max_links]

In [ ]:
def remove_unwanted_lines(paragraphs):
    bad_starts = [
        "Xem thêm",
        "Mời độc giả",
        "Độc giả gửi bài",
        "Copy link",
        "Trở lại",
        "Theo ",
    ]

    bad_contains = [
        "Bấm để xem",
        "Ảnh:",
        "Video:",
        "Nguồn:",
        "Xem video",
        "Đọc thêm",
        "Tại đây",
    ]

    cleaned = []
    for p in paragraphs:
        p = normalize_text(p)
        if not p:
            continue
        if len(p) < 20:
            continue
        if any(p.startswith(x) for x in bad_starts):
            continue
        if any(x.lower() in p.lower() for x in bad_contains):
            continue
        cleaned.append(p)

    unique = []
    seen = set()
    for p in cleaned:
        if p not in seen:
            unique.append(p)
            seen.add(p)

    return unique

def cut_after_stop_markers(text):
    stop_markers = [
        "Copy link",
        "Mời độc giả",
        "Xem thêm",
        "Đọc thêm",
        "Bình luận",
    ]
    for marker in stop_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
    return normalize_text(text)

In [ ]:
def parse_vnexpress_article(url):
    res = requests.get(url, headers=random_headers(), timeout=20)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "lxml")

    # title
    title = ""
    title_tag = soup.find("h1")
    if title_tag:
        title = normalize_text(title_tag.get_text(" ", strip=True))

    # publish time
    publish_time = ""
    time_tag = soup.find("span", class_=re.compile(r"date|time", re.I))
    if time_tag:
        publish_time = normalize_text(time_tag.get_text(" ", strip=True))
    else:
        meta_time = soup.find("meta", attrs={"property": "article:published_time"})
        if meta_time and meta_time.get("content"):
            publish_time = meta_time["content"].strip()

    # category
    category = ""
    breadcrumb = soup.find(["ul", "div"], class_=re.compile(r"breadcrumb|header-top|parent-cate", re.I))
    if breadcrumb:
        category = normalize_text(breadcrumb.get_text(" ", strip=True))

    if not category:
        meta_section = soup.find("meta", attrs={"name": "its_category"})
        if meta_section and meta_section.get("content"):
            category = meta_section["content"].strip()

    # description
    description = ""
    desc_tag = soup.find("p", class_=re.compile(r"description|lead", re.I))
    if desc_tag:
        description = normalize_text(desc_tag.get_text(" ", strip=True))
    else:
        meta_desc = soup.find("meta", attrs={"name": "description"})
        if meta_desc and meta_desc.get("content"):
            description = normalize_text(meta_desc["content"])

    # article body
    article_body = (
        soup.find("article")
        or soup.find("div", class_=re.compile(r"fck_detail|sidebar-1|content-detail|article-content", re.I))
        or soup.find("section", class_=re.compile(r"section page-detail", re.I))
    )

    paragraphs = []

    if article_body:
        for p in article_body.find_all("p"):
            txt = normalize_text(p.get_text(" ", strip=True))
            if txt:
                paragraphs.append(txt)
    else:
        for p in soup.find_all("p"):
            txt = normalize_text(p.get_text(" ", strip=True))
            if txt:
                paragraphs.append(txt)

    paragraphs = remove_unwanted_lines(paragraphs)

    final_paragraphs = []
    seen = set()
    for p in paragraphs:
        if p == title:
            continue
        if description and p == description:
            continue
        if p not in seen:
            final_paragraphs.append(p)
            seen.add(p)

    content = "\n".join(final_paragraphs)
    content = cut_after_stop_markers(content)

    return {
        "url": url,
        "title": title,
        "publish_time": publish_time,
        "category": category,
        "description": description,
        "content": content,
        "source_word_count": word_count(content),
        "numbers_in_source": extract_numbers(content)
    }

In [ ]:
def generate_section_urls(section, max_pages=10):
    urls = [section]
    for p in range(2, max_pages + 1):
        urls.append(f"{section}-p{p}")
    return urls

sections = [
    "https://vnexpress.net/kinh-doanh",
    "https://vnexpress.net/kinh-doanh/doanh-nghiep",
    "https://vnexpress.net/kinh-doanh/chung-khoan",
    "https://vnexpress.net/kinh-doanh/ebank",
    "https://vnexpress.net/kinh-doanh/vi-mo",
    "https://vnexpress.net/kinh-doanh/hang-hoa",
]

In [ ]:
def get_links_from_one_section(
    section,
    max_pages=10,
    max_links_per_page=100,
    max_empty_pages=3,
    sleep_sec=1.0
):
    section_urls = generate_section_urls(section, max_pages=max_pages)
    section_links = set()
    empty_page_count = 0

    print(f"\n===== CRAWL SECTION: {section} =====")

    for i, list_url in enumerate(section_urls, 1):
        try:
            links = get_article_links(list_url, max_links=max_links_per_page)

            before = len(section_links)
            section_links.update(links)
            after = len(section_links)
            new_links = after - before

            print(f"[{section} | PAGE {i}] {list_url}")
            print(f"  - Link lấy được từ page này: {len(links)}")
            print(f"  - Link mới thêm vào section: {new_links}")
            print(f"  - Tổng link của section: {after}")

            if len(links) == 0:
                empty_page_count += 1
            else:
                empty_page_count = 0

            if empty_page_count >= max_empty_pages:
                print(f"Section này có {max_empty_pages} page rỗng liên tiếp. Dừng section.")
                break

            time.sleep(sleep_sec)

        except Exception as e:
            print(f"[ERROR] {list_url} -> {e}")
            empty_page_count += 1
            if empty_page_count >= max_empty_pages:
                print(f"Section này lỗi/rỗng liên tiếp. Dừng section.")
                break

    return sorted(section_links)

In [ ]:
def collect_links_from_sections(
    sections,
    max_total_links=700,
    max_pages_per_section=10,
    max_links_per_page=100,
    max_empty_pages=3,
    sleep_sec=1.0
):
    all_links = set()

    for section in sections:
        section_links = get_links_from_one_section(
            section=section,
            max_pages=max_pages_per_section,
            max_links_per_page=max_links_per_page,
            max_empty_pages=max_empty_pages,
            sleep_sec=sleep_sec
        )

        before = len(all_links)
        all_links.update(section_links)
        after = len(all_links)

        print(f"\n>>> Gộp xong section: {section}")
        print(f"  - Link mới thêm vào tổng: {after - before}")
        print(f"  - Tổng link hiện có: {after}")

        if len(all_links) >= max_total_links:
            print(f"Đã đủ {max_total_links} link. Dừng gom link.")
            break

    return sorted(all_links)[:max_total_links]

In [ ]:
all_links = collect_links_from_sections(
    sections=sections,
    max_total_links=700,       # khống chế tổng link ở đây
    max_pages_per_section=12,  # tăng/giảm tùy nhu cầu
    max_links_per_page=100,
    max_empty_pages=3,
    sleep_sec=1.0
)

print("Tổng số link đã gom:", len(all_links))
print(all_links[:10])


===== CRAWL SECTION: https://vnexpress.net/kinh-doanh =====
[https://vnexpress.net/kinh-doanh | PAGE 1] https://vnexpress.net/kinh-doanh
  - Link lấy được từ page này: 50
  - Link mới thêm vào section: 50
  - Tổng link của section: 50
[https://vnexpress.net/kinh-doanh | PAGE 2] https://vnexpress.net/kinh-doanh-p2
  - Link lấy được từ page này: 30
  - Link mới thêm vào section: 30
  - Tổng link của section: 80
[https://vnexpress.net/kinh-doanh | PAGE 3] https://vnexpress.net/kinh-doanh-p3
  - Link lấy được từ page này: 30
  - Link mới thêm vào section: 30
  - Tổng link của section: 110
[https://vnexpress.net/kinh-doanh | PAGE 4] https://vnexpress.net/kinh-doanh-p4
  - Link lấy được từ page này: 30
  - Link mới thêm vào section: 30
  - Tổng link của section: 140
[https://vnexpress.net/kinh-doanh | PAGE 5] https://vnexpress.net/kinh-doanh-p5
  - Link lấy được từ page này: 30
  - Link mới thêm vào section: 30
  - Tổng link của section: 170
[https://vnexpress.net/kinh-doanh | PAGE 6] https

In [ ]:
def crawl_articles_from_links(
    links,
    min_words=120,
    min_numbers=2,
    sleep_sec=1.0
):
    rows = []
    seen = set()

    for i, url in enumerate(links, 1):
        try:
            row = parse_vnexpress_article(url)

            if not row["title"] or not row["content"]:
                print(f"[SKIP] Bài rỗng: {url}")
                continue

            if row["source_word_count"] < min_words:
                print(f"[SKIP] Quá ngắn ({row['source_word_count']} từ): {row['title'][:80]}")
                continue

            if len(row["numbers_in_source"]) < min_numbers:
                print(f"[SKIP] Ít số liệu ({len(row['numbers_in_source'])}): {row['title'][:80]}")
                continue

            dup_key = (
                row["title"].strip().lower(),
                row["content"][:500].strip().lower()
            )
            if dup_key in seen:
                print(f"[SKIP] Trùng: {row['title'][:80]}")
                continue

            seen.add(dup_key)

            row["numbers_in_source"] = json.dumps(row["numbers_in_source"], ensure_ascii=False)
            rows.append(row)

            print(f"[OK {len(rows)}] {row['title'][:90]}")
            time.sleep(sleep_sec)

        except Exception as e:
            print(f"[ERROR ARTICLE {i}] {url} -> {e}")

    return pd.DataFrame(rows)

In [ ]:
df = crawl_articles_from_links(
    links=all_links,
    min_words=120,
    min_numbers=2,
    sleep_sec=1.0
)

print("Số bài hợp lệ cuối cùng:", len(df))
df.head()

[OK 1] 10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuất thực phẩm
[OK 2] 10 triệu trái phiếu DNSE chính thức lên sàn HNX
[OK 3] 11 điểm đổ xăng thay thế các trạm đóng cửa trên cao tốc Long Thành - Dầu Giây
[OK 4] 11 đơn vị xin cấp phép sản xuất vàng miếng
[OK 5] 15 tiêu chí chọn cổ phiếu của Philip Fisher
[OK 6] 17 cá nhân bị cấm giao dịch chứng khoán hai năm
[OK 7] 2025 - năm 'thanh lọc' nhà bán hàng trên sàn thương mại điện tử
[OK 8] 247Express ghi lại hành trình thiện nguyện bằng âm nhạc
[OK 9] 28 cổ phiếu Việt Nam có thể vào rổ chỉ số FTSE sau khi nâng hạng
[OK 10] 3 yếu tố tác động đến thị trường chứng khoán cuối năm
[OK 11] 33 'cá mập' sẽ mua hơn 264 triệu cổ phiếu BIDV
[OK 12] 4 nhóm ngành tiềm năng hút dòng tiền chứng khoán năm 2026
[OK 13] 75 doanh nhân trúng cử đại biểu HĐND cấp tỉnh
[OK 14] 9 công ty Việt vào top 500 tốt nhất châu Á - Thái Bình Dương 2026
[OK 15] ABBank lên kế hoạch tăng tốc trong năm 2026
[OK 16] ACB giảm 2% lãi vay một năm cho hộ kinh doanh
[OK 17] ACB tung

,url,title,publish_time,category,description,content,source_word_count,numbers_in_source
0,https://vnexpress.net/10-nam-c-p-cu-chi-chuan-...,10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuấ...,"Thứ ba, 21/4/2026",Kinh doanh Doanh nghiệp,Nhà máy Củ Chi chọn hướng sản xuất có trách nh...,"Lễ kỷ niệm chủ đề ""Nhà máy C.P. Củ Chi: 10 năm...",820,"[""10 năm"", ""26"", ""3"", ""10 năm"", ""10 năm"", ""63...."
1,https://vnexpress.net/10-trieu-trai-phieu-dnse...,10 triệu trái phiếu DNSE chính thức lên sàn HNX,"Thứ ba, 21/4/2026",Kinh doanh Chứng khoán,Lô trái phiếu DSE125018 trị giá 1.000 tỷ ...,10 triệu trái phiếu doanh nghiệp niêm yết có k...,555,"[""10 triệu"", ""24 tháng"", ""8,3"", ""3,5"", ""6 thán..."
2,https://vnexpress.net/11-diem-do-xang-thay-the...,11 điểm đổ xăng thay thế các trạm đóng cửa trê...,"Thứ ba, 21/4/2026",Kinh doanh Hàng hóa,Sở Công Thương TP HCM công bố 11 cửa hàng xăng...,"Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...",357,"[""51"", ""41"", ""100"", ""24"", ""24"", ""41"", ""100""]"
3,https://vnexpress.net/11-don-vi-xin-cap-phep-s...,11 đơn vị xin cấp phép sản xuất vàng miếng,"Thứ ba, 21/4/2026",Kinh doanh Ebank Ngân hàng,Ngân hàng Nhà nước cho biết đã nhận 11 hồ sơ đ...,"Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...",404,"[""14"", ""4"", ""1.000 tỷ"", ""50.000 tỷ"", ""38"", ""1...."
4,https://vnexpress.net/15-tieu-chi-chon-co-phie...,15 tiêu chí chọn cổ phiếu của Philip Fisher,"Thứ ba, 21/4/2026",Kinh doanh Chứng khoán,"Philip Fisher, huyền thoại chứng khoán Mỹ, đưa...",Philip Fisher (1907-2004) là một trong những n...,949,"[""15"", ""4""]"


In [ ]:
output_file = "vnexpress_kinhdoanh_finance_related.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print("Đã lưu file:", output_file)

Đã lưu file: vnexpress_kinhdoanh_finance_related.csv


In [ ]:
from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Chỉ giữ lại 2 cột title, content
df = df[["title", "content"]].copy()

# Chuẩn hóa text nhẹ
import re

def clean_whitespace(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

def normalize_text(text):
    text = clean_whitespace(text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip()

df["title"] = df["title"].fillna("").astype(str).map(normalize_text)
df["content"] = df["content"].fillna("").astype(str).map(normalize_text)

# Bỏ dòng rỗng
df = df[(df["title"] != "") & (df["content"] != "")].copy()

# Bỏ trùng
df = df.drop_duplicates(subset=["title", "content"]).reset_index(drop=True)

# Thêm cột summary rỗng
df["summary"] = ""

print("Kích thước dữ liệu sau khi rút gọn:", df.shape)
df.head()

Kích thước dữ liệu sau khi rút gọn: (661, 3)


,title,content,summary
0,10 năm C.P. Củ Chi chuẩn hóa quy trình sản xuấ...,"Lễ kỷ niệm chủ đề ""Nhà máy C.P. Củ Chi: 10 năm...",
1,10 triệu trái phiếu DNSE chính thức lên sàn HNX,10 triệu trái phiếu doanh nghiệp niêm yết có k...,
2,11 điểm đổ xăng thay thế các trạm đóng cửa trê...,"Các điểm thay thế nằm dọc Xa lộ Hà Nội, Nguyễn...",
3,11 đơn vị xin cấp phép sản xuất vàng miếng,"Tại họp báo chiều 14/4, ông Đào Xuân Tuấn, Vụ ...",
4,15 tiêu chí chọn cổ phiếu của Philip Fisher,Philip Fisher (1907-2004) là một trong những n...,


In [ ]:
output_file = "dataset_title_content_summary.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Đã lưu file:", output_file)
print("Số dòng:", len(df))

Đã lưu file: dataset_title_content_summary.csv
Số dòng: 661


In [ ]:
from google.colab import files
files.download("dataset_title_content_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>